In [1]:
import os
import sys

import math
import time
import datetime
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import torch
from torch.utils.data import Dataset
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
from matcho import Unet2D
# from YourDataset import YourDataset  # Import your custom dataset here
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
from torchinfo import summary
import torchprofile

import pickle

torch.manual_seed(23)

scaler = GradScaler()

DTYPE = torch.float32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 200
plt.rcParams["font.family"] = "serif"

import scipy.stats as stats

Using device: cuda


In [2]:
# Define your custom loss function here
class CustomLoss(nn.Module):
    def __init__(self):
        super(CustomLoss, self).__init__()

    def forward(self, y_pred, y_true):
        # Implement your custom loss calculation here
        # loss = torch.mean((y_pred - y_true) ** 2)  # Example: Mean Squared Error
        loss = torch.norm(y_true-y_pred, p=2)/torch.norm(y_true, p=2)
        return loss

class YourDataset(Dataset):
    def __init__(self, x, t, y, transform=None):
        self.x = x
        self.t = t
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x_sample = self.x[idx]
        t_sample = self.t[idx]
        y_sample = self.y[idx]

        if self.transform:
            x_sample, t_sample, y_sample = self.transform(x_sample, t_sample, y_sample)

        return x_sample, t_sample, y_sample


def preprocess(traj, Par):
    x = sliding_window_view(traj[:,:-(Par['lf']-1),:,:], window_shape=Par['lb'], axis=1 ).transpose(0,1,4,2,3).reshape(-1,Par['lb'],Par['nx'], Par['ny'])
    y = sliding_window_view(traj[:,Par['lb']-1:,:,:], window_shape=Par['lf'], axis=1 ).transpose(0,1,4,2,3).reshape(-1,Par['lf'],Par['nx'], Par['ny'])
    t = np.linspace(0,1,Par['lf']).reshape(-1,1)

    nt = y.shape[1]
    n_samples = y.shape[0]

    t = np.tile(t, [n_samples,1]).reshape(-1,)                                                     #[_*nt, ]
    x = np.repeat(x,nt, axis=0)                                   #[_*nt, 1, 64, 64]
    y = y.reshape(y.shape[0]*y.shape[1],1,y.shape[2],y.shape[3])  #[_*nt, 64, 64]


    print('x: ', x.shape)
    print('y: ', y.shape)
    print('t: ', t.shape)
    print()
    return x,y,t

In [ ]:
# Load your data into NumPy arrays (x_train, t_train, y_train, x_val, t_val, y_val, x_test, t_test, y_test)
#########################
res = 128
begin_time = time.time()
traj = np.load(f"/oscar/data/gk/voommen/no_diffusion/kolmogrov/data/alpha_1.5_tau_14_re_2007_N_1000_T_50_nt_200_nx_512/res_{res}/traj.npy") #[1000, nx, ny, nt]
traj = traj.transpose(0,3,1,2)
print(f"Data Loading Time: {time.time() - begin_time:.1f}s")

traj_train = traj[:800, :80][:, ::2]
traj_val   = traj[800:900, :80][:, ::2]
traj_test  = traj[900:, :80][:, ::2]

print(f"traj_train: {traj_train.shape}")
print(f"traj_val: {traj_val.shape}")
print(f"traj_test: {traj_test.shape}")

Par = {}
# Par['nt'] = 100 
Par['nx'] = traj_train.shape[2]
Par['ny'] = traj_train.shape[3]
Par['nf'] = 1
Par['d_emb'] = 128

Par['lb'] = 20
Par['lf'] = 20+1 
# Par['temp'] = Par['nt'] - Par['lb'] - Par['lf'] + 2

Par['num_epochs'] = 500

begin_time = time.time()
print('\nTrain Dataset')
x_train, y_train, t_train = preprocess(traj_train, Par)
print('\nValidation Dataset')
x_val, y_val, t_val  = preprocess(traj_val, Par)
print('\nTest Dataset')
x_test, y_test, t_test  = preprocess(traj_test, Par)
print(f"Data Preprocess Time: {time.time() - begin_time:.1f}s")

t_min = np.min(t_train)
t_max = np.max(t_train)

Par['inp_shift'] = np.mean(x_train) 
Par['inp_scale'] = np.std(x_train)
Par['out_shift'] = np.mean(y_train)
Par['out_scale'] = np.std(y_train)
Par['t_shift']   = t_min
Par['t_scale']   = t_max - t_min

with open('Par.pkl', 'wb') as f:
    pickle.dump(Par, f)

# sys.exit()
#########################

# Create custom datasets
x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
t_train_tensor = torch.tensor(t_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

x_val_tensor   = torch.tensor(x_val,   dtype=torch.float32)
t_val_tensor   = torch.tensor(t_val,   dtype=torch.float32)
y_val_tensor   = torch.tensor(y_val,   dtype=torch.float32)

x_test_tensor  = torch.tensor(x_test,  dtype=torch.float32)
t_test_tensor  = torch.tensor(t_test,  dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.float32)

train_dataset = YourDataset(x_train_tensor, t_train_tensor, y_train_tensor)
val_dataset = YourDataset(x_val_tensor, t_val_tensor, y_val_tensor)
test_dataset = YourDataset(x_test_tensor, t_test_tensor, y_test_tensor)

# Define data loaders
train_batch_size = 20 #100
val_batch_size   = 20 #100
test_batch_size  = 20 #100
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=val_batch_size)
test_loader = DataLoader(test_dataset, batch_size=test_batch_size)

Data Loading Time: 3.9s
traj_train: (800, 40, 128, 128)
traj_val: (100, 40, 128, 128)
traj_test: (100, 40, 128, 128)

Train Dataset
x:  (16800, 20, 128, 128)
y:  (16800, 1, 128, 128)
t:  (16800,)


Validation Dataset
x:  (2100, 20, 128, 128)
y:  (2100, 1, 128, 128)
t:  (2100,)


Test Dataset
x:  (2100, 20, 128, 128)
y:  (2100, 1, 128, 128)
t:  (2100,)

Data Preprocess Time: 7.1s


In [ ]:
model = Unet2D(dim=16, channels=Par['lb'], Par=Par, dim_mults=(1, 2, 4, 8)).to(device).to(torch.float32)

path_model = 'models/best_model.pt'
model.load_state_dict(torch.load(path_model))

print(summary(model, input_size=((1,)+x_train.shape[1:], (1,)) ) )

# Adjust the dimensions as per your model's input size
dummy_x = torch.tensor(torch.randn(20, 20, 128,128),   dtype=DTYPE, device=device)
dummy_t = torch.tensor(torch.randn(20,),   dtype=DTYPE, device=device)
dummy_input = (dummy_x, dummy_t)

# Profile the model
flops = torchprofile.profile_macs(model, dummy_input)
print(f"FLOPs: {flops:.3e}")

# Define loss function and optimizer
criterion = CustomLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Speed TEST

In [8]:
model.eval()
test_loss = 0.0
with torch.no_grad():
    for x, t, y_true in test_loader:
        begin_time = time.time()
        with autocast():
            y_pred = model(x.to(device), t.to(device))
        print(y_pred.shape)
        print(f"elapsed time: {time.time() - begin_time:.4f}")  

torch.Size([20, 1, 128, 128])
elapsed time: 0.0174
torch.Size([20, 1, 128, 128])
elapsed time: 0.0164
torch.Size([20, 1, 128, 128])
elapsed time: 0.0162
torch.Size([20, 1, 128, 128])
elapsed time: 0.0165
torch.Size([20, 1, 128, 128])
elapsed time: 0.0163
torch.Size([20, 1, 128, 128])
elapsed time: 0.0160
torch.Size([20, 1, 128, 128])
elapsed time: 0.0162
torch.Size([20, 1, 128, 128])
elapsed time: 0.0161
torch.Size([20, 1, 128, 128])
elapsed time: 0.0160
torch.Size([20, 1, 128, 128])
elapsed time: 0.0160
torch.Size([20, 1, 128, 128])
elapsed time: 0.0160
torch.Size([20, 1, 128, 128])
elapsed time: 0.0165
torch.Size([20, 1, 128, 128])
elapsed time: 0.0159
torch.Size([20, 1, 128, 128])
elapsed time: 0.0159
torch.Size([20, 1, 128, 128])
elapsed time: 0.0160
torch.Size([20, 1, 128, 128])
elapsed time: 0.0160
torch.Size([20, 1, 128, 128])
elapsed time: 0.0159
torch.Size([20, 1, 128, 128])
elapsed time: 0.0159
torch.Size([20, 1, 128, 128])
elapsed time: 0.0159
torch.Size([20, 1, 128, 128])
e

# Sanity Check

In [11]:
y_true_ls = []
y_pred_ls = []

model.eval()
train_loss = 0.0
with torch.no_grad():
    for x, t, y_true in train_loader:
        with autocast():
            y_pred = model(x.to(device), t.to(device))
            loss   = criterion(y_pred, y_true.to(device))
        train_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

train_loss /= len(train_loader)
print(f"Train Loss: {train_loss:.4e}")

TRAIN_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
TRAIN_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"TRAIN_TRUE: {TRAIN_TRUE.shape}, DTYPE: {TRAIN_TRUE.dtype}")
print(f"TRAIN_PRED: {TRAIN_PRED.shape}, DTYPE: {TRAIN_PRED.dtype}")



y_true_ls = []
y_pred_ls = []

model.eval()
val_loss = 0.0
with torch.no_grad():
    for x, t, y_true in val_loader:
        with autocast():
            y_pred = model(x.to(device), t.to(device))
            loss   = criterion(y_pred, y_true.to(device))
        val_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

val_loss /= len(val_loader)
print(f"Val Loss: {val_loss:.4e}")

VAL_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
VAL_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"VAL_TRUE: {VAL_TRUE.shape}, DTYPE: {VAL_TRUE.dtype}")
print(f"VAL_PRED: {VAL_PRED.shape}, DTYPE: {VAL_PRED.dtype}")



y_true_ls = []
y_pred_ls = []

model.eval()
test_loss = 0.0
with torch.no_grad():
    for x, t, y_true in test_loader:
        with autocast():
            y_pred = model(x.to(device), t.to(device))
            loss   = criterion(y_pred, y_true.to(device))
        test_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

test_loss /= len(test_loader)
print(f"Test Loss: {test_loss:.4e}")

TEST_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
TEST_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"TEST_TRUE: {TEST_TRUE.shape}, DTYPE: {TEST_TRUE.dtype}")
print(f"TEST_PRED: {TEST_PRED.shape}, DTYPE: {TEST_PRED.dtype}")

Train Loss: 1.0535e-01
TRAIN_TRUE: (800, 21, 128, 128), DTYPE: float32
TRAIN_PRED: (800, 21, 128, 128), DTYPE: float32
Val Loss: 2.2408e-01
VAL_TRUE: (100, 21, 128, 128), DTYPE: float32
VAL_PRED: (100, 21, 128, 128), DTYPE: float32
Test Loss: 2.3557e-01
TEST_TRUE: (100, 21, 128, 128), DTYPE: float32
TEST_PRED: (100, 21, 128, 128), DTYPE: float32


In [12]:
np.save("TRAIN_TRUE.npy", TRAIN_TRUE[:,1:])
np.save("TRAIN_PRED.npy", TRAIN_PRED[:,1:])

np.save("VAL_TRUE.npy", VAL_TRUE[:,1:])
np.save("VAL_PRED.npy", VAL_PRED[:,1:])

np.save("TEST_TRUE.npy", TEST_TRUE[:,1:])
np.save("TEST_PRED.npy", TEST_PRED[:,1:])